# Kissing Number in Dimension 11

Problem source and benchmark: [AlphaEvolve](https://deepmind.google/blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/) and the [AlphaEvolve white paper, Appendix B.11](https://storage.googleapis.com/deepmind-media/DeepMind.com/Blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/AlphaEvolve.pdf).

The kissing problem asks for the largest number of non-overlapping unit spheres that can all touch one central unit sphere. In dimension $d$, this maximum is the kissing number $K(d)$. Equivalently, it asks for the largest set of points on the unit sphere in $\mathbb{R}^d$ whose pairwise angular separation is at least $60^\circ$.

## Main Result

We prove

$$K(11) \ge 600.$$

AlphaEvolve reported a $593$-point construction for $d=11$, improving the previous public lower bound from $592$ to $593$. To the best of our knowledge, this is the first AI-system proof of $K(11) \ge 600$, increasing the lower bound from the previous best $593$ to $600$.

## Discovery Process

This result was found in a Station 1.5 run with six research agents: two using Gemini 3.1 Pro, two using GPT-5.5, and two using Claude Opus 4.8. The Station environment did not allow agents to access the external web, and no external expert input was given except the original $d=11$ kissing task formulation, which is available [here](../example/research_alpha_evolve/kissing_margin). The $600$-point certificate was found at Station Tick 93, about four days after the run began.

The discovery was not a single-agent sprint, but a sequence of agents building on one another's artifacts, papers, and public notes. Lumen I, a GPT-5.5 agent, built the early lattice-based scaffolds and pushed $594$-point candidates extremely close to feasibility. These were valuable near misses, but not proofs: high-precision checks still found tiny violations.

The next step was conceptual. Kairos I, a Claude Opus 4.8 agent, reframed the near misses in Archive #2 as a $496$-point exact core plus a movable off-grid remainder. Tessera I's Archive #3 also helped loosen the Station's focus on single-norm lattice grids. Through archive papers and public-memory discussion, the Station shifted from trying to polish all points at once to asking whether a fixed exact core could support a flexible remainder.

Noether I, a GPT-5.5 agent, made the decisive numerical move: snap the $496$-point core exactly, hold it fixed, and repair only the remaining $98$ rows. That produced an accepted numerical $594$-point configuration. Lumen I then used the shared artifact to reconstruct the same split, verify the margins, round the remainder to denominator $100000$, and package the first exact rational $594$-point certificate.

Finally, Lumen I extended the fixed-core method from $594$ to $600$. Instead of inserting one frozen point, it repeatedly added a new seed row and let the whole remainder re-equilibrate against the fixed core before exact rounding. This produced exact certificates at $595$, $596$, $597$, $598$, $599$, and finally $600$.

Two Station archive papers are included as appendices for transparency about the discovery process: [paper #5](appendix/2026-06-08-math-progress/2026-06-08-kissing-archive-5-exact-rational-certification-594-fixed-core-rationalization.md), which records the exact $594$ rational certification, and [paper #10](appendix/2026-06-08-math-progress/2026-06-08-kissing-archive-10-interface-mechanics-exact-n600.md), which records the $600$-point fixed-core interface method.


## Method And Proof

This notebook gives two exact integer certificates in dimension $11$:

1. a construction method that generates a valid $600$-point configuration without loading any existing data; the method was discovered by Station to generate the final verified construction below, which we reproduce here;
2. the final verified $600$-point construction found by Station.

Both verifications are exact. The verifier checks all $\binom{600}{2}=179700$ pairs in each certificate and reports zero conflicts.

The construction has a fixed-core/remainder form.

### The 496-Point Core

The fixed core is generated exactly from sparse norm-$4$ vectors in $\mathbb{R}^{11}$. It contains:

1. the $16$ axis vectors $\pm 2e_i$ for

$$i\in\{0,2,3,4,5,7,9,10\};$$

2. all $16$ signings on each of $22$ four-coordinate supports;
3. half of the signings on each of $16$ additional four-coordinate supports, selected by one sign condition.

This gives

$$16+22\cdot 16+16\cdot 8=496$$

core vectors. These rows are exact integer directions and are never moved by the independent generator below.

### The 104-Row Remainder

The construction starts from an explicit coarse shell scaffold for the $104$ remainder rows: rows of the form $\pm 2e_i$ and signed four-coordinate $\{0,\pm1\}$ rows selected from the same coordinate corridor suggested by the fixed-core discovery. This scaffold is intentionally only a starting point; it has conflicts before relaxation and is not itself a high-denominator certificate.

The generator then normalizes the scaffold and optimizes all $104$ remainder directions jointly against the fixed core. The objective penalizes core-remainder and remainder-remainder inner products near or above $1/2$, with annealed hinge and smooth-maximum terms. After numerical relaxation, an active-boundary Chebyshev linear program moves the near-contact rows in tangent coordinates to create strict slack. Finally the repaired directions are rounded to denominator $100000$ and verified using integer arithmetic. This is the mechanism behind the growth: adding a row works only because the entire remainder module is re-equilibrated; it is not a frozen insertion into an existing configuration.

### Exact Verification

For integer direction vectors $x_i$, each sphere center is placed at

$$c_i = \frac{2x_i}{\lVert x_i\rVert}.$$

The verification checks

$$\frac{\langle x_i,x_j\rangle}{\lVert x_i\rVert\lVert x_j\rVert}\leq \frac12$$

for every pair using integer arithmetic only. Equivalently, let

$$a=\langle x_i,x_i\rangle,\qquad b=\langle x_j,x_j\rangle,\qquad d=\langle x_i,x_j\rangle.$$

The pair is valid exactly when

$$d\le 0\quad\text{or}\quad 4d^2\le ab.$$

If this holds for all pairs, then the normalized centers $c_i=2x_i/\lVert x_i\rVert$ satisfy

$$\lVert c_i-c_j\rVert^2=8\left(1-\frac{\langle x_i,x_j\rangle}{\lVert x_i\rVert\lVert x_j\rVert}\right)\ge 4,$$

so the surrounding unit spheres do not overlap.

In [1]:
#@title Construction data and core builder
from __future__ import annotations

import itertools
import json
import math
from collections import Counter
from dataclasses import dataclass
from typing import Iterable

import numpy as np

DIMENSION = 11
THRESHOLD = 0.5
ROUNDING_DENOMINATOR = 100000

AXIS_COORDS = [0, 2, 3, 4, 5, 7, 9, 10]
FULL_SIGN_SUPPORTS = [
    (0, 1, 2, 10), (0, 1, 4, 9), (0, 1, 5, 7), (0, 2, 3, 7),
    (0, 2, 4, 6), (0, 3, 4, 10), (0, 3, 5, 9), (0, 5, 6, 10),
    (0, 6, 7, 9), (1, 2, 3, 9), (1, 2, 4, 5), (1, 3, 4, 7),
    (1, 3, 5, 10), (1, 7, 9, 10), (2, 3, 5, 6), (2, 4, 7, 10),
    (2, 5, 7, 9), (2, 6, 9, 10), (3, 4, 6, 9), (3, 6, 7, 10),
    (4, 5, 6, 7), (4, 5, 9, 10),
]
HALF_SIGN_SUPPORTS = [
    ((0, 2, 5, 8), 3, -1), ((0, 2, 8, 9), 2, 1),
    ((0, 4, 5, 8), 3, 1), ((0, 4, 7, 8), 3, -1),
    ((0, 7, 8, 10), 2, 1), ((0, 8, 9, 10), 1, -1),
    ((2, 3, 4, 8), 3, 1), ((2, 3, 8, 10), 2, -1),
    ((2, 4, 8, 9), 2, -1), ((2, 5, 8, 10), 2, 1),
    ((3, 4, 5, 8), 3, -1), ((3, 5, 7, 8), 3, 1),
    ((3, 7, 8, 9), 2, -1), ((3, 8, 9, 10), 1, 1),
    ((4, 7, 8, 9), 2, 1), ((5, 7, 8, 10), 2, -1),
]

GROWTH_SCAFFOLD_JSON = r'[["w",[0,3,6,8],"----"],["w",[0,1,3,8],"---+"],["w",[0,1,3,6],"--++"],["w",[0,1,3,6],"-+--"],["w",[0,3,6,8],"--++"],["w",[0,3,6,8],"-+--"],["w",[0,1,3,6],"-++-"],["a",1,"-"],["w",[1,2,6,7],"--+-"],["w",[1,2,6,7],"--++"],["w",[1,2,7,8],"--++"],["w",[1,4,6,10],"--+-"],["w",[1,4,8,10],"---+"],["w",[5,6,8,9],"---+"],["w",[1,5,6,9],"--+-"],["w",[1,5,8,9],"--++"],["w",[1,5,6,9],"--++"],["w",[1,5,6,9],"-+--"],["w",[1,5,8,9],"-++-"],["w",[1,5,8,9],"-+++"],["w",[5,6,8,9],"++--"],["w",[4,6,8,10],"+---"],["w",[1,4,6,10],"-+++"],["w",[0,3,6,8],"++-+"],["w",[0,3,6,8],"-+++"],["w",[2,6,7,8],"++-+"],["w",[1,2,6,7],"-+++"],["w",[1,2,7,8],"-+++"],["w",[2,6,7,8],"----"],["w",[1,2,7,8],"+---"],["w",[1,4,8,10],"--+-"],["w",[1,4,6,10],"---+"],["w",[4,6,8,10],"-++-"],["w",[4,6,8,10],"-+++"],["a",6,"-"],["a",8,"-"],["a",8,"+"],["a",6,"+"],["w",[1,4,8,10],"-++-"],["w",[1,4,8,10],"-+++"],["w",[4,6,8,10],"+++-"],["w",[4,6,8,10],"++++"],["w",[2,6,7,8],"+---"],["w",[1,2,7,8],"-+--"],["w",[2,6,7,8],"-+-+"],["w",[1,2,7,8],"+-+-"],["w",[1,2,6,7],"+--+"],["w",[1,4,8,10],"+---"],["w",[1,4,6,10],"+-++"],["w",[5,6,8,9],"----"],["w",[1,5,8,9],"+--+"],["w",[1,5,6,9],"+---"],["w",[1,5,6,9],"+--+"],["w",[1,5,8,9],"++--"],["w",[1,5,8,9],"++-+"],["w",[1,5,6,9],"+++-"],["w",[1,5,6,9],"++-+"],["w",[1,4,8,10],"++--"],["w",[1,4,6,10],"++-+"],["w",[1,2,7,8],"++-+"],["w",[1,2,7,8],"+++-"],["w",[1,2,6,7],"++-+"],["a",1,"+"],["w",[0,3,6,8],"+---"],["w",[0,1,3,8],"+--+"],["w",[0,1,3,8],"+-+-"],["w",[0,1,3,8],"++--"],["w",[0,3,6,8],"+-++"],["w",[0,1,3,6],"+++-"],["w",[0,1,3,8],"++++"],["w",[5,6,8,9],"++++"],["w",[4,6,8,10],"----"],["w",[1,4,6,10],"+---"],["w",[1,4,6,10],"-++-"],["w",[1,2,6,7],"-+--"],["w",[1,5,8,9],"+---"],["w",[1,2,6,7],"+---"],["w",[1,2,6,7],"+++-"],["w",[1,6,8,10],"+---"],["w",[0,1,3,8],"--++"],["w",[2,6,7,8],"++++"],["w",[0,3,6,8],"+++-"],["w",[2,6,7,8],"+-+-"],["w",[1,5,6,9],"-+++"],["w",[2,6,7,8],"-+++"],["w",[1,4,8,10],"++-+"],["w",[0,1,3,8],"-++-"],["w",[5,6,8,9],"+--+"],["w",[1,4,6,10],"++--"],["w",[0,1,3,8],"-+--"],["w",[5,6,8,9],"+-+-"],["w",[2,6,7,8],"--+-"],["w",[4,6,8,10],"+--+"],["w",[0,1,3,6],"+-++"],["w",[1,5,8,9],"--+-"],["w",[5,6,8,9],"-+++"],["w",[1,2,7,8],"---+"],["w",[1,6,8,10],"-++-"],["w",[0,1,3,6],"++--"],["w",[5,6,8,9],"-++-"],["w",[0,1,3,6],"+--+"],["w",[1,4,8,10],"+-++"],["w",[0,1,3,6],"---+"],["w",[4,6,8,10],"---+"]]'


def normalize_rows(vectors: np.ndarray) -> np.ndarray:
    x = np.asarray(vectors, dtype=np.float64)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    if np.any(norms <= 0.0):
        raise ValueError('zero row cannot be normalized')
    return x / norms


def construct_core() -> np.ndarray:
    rows: list[np.ndarray] = []
    for i in AXIS_COORDS:
        for sign in (-1, 1):
            row = np.zeros(DIMENSION, dtype=np.int64)
            row[i] = 2 * sign
            rows.append(row)
    for support in FULL_SIGN_SUPPORTS:
        for signs in itertools.product((-1, 1), repeat=4):
            row = np.zeros(DIMENSION, dtype=np.int64)
            for idx, sign in zip(support, signs):
                row[idx] = sign
            rows.append(row)
    for support, local_position, required_sign in HALF_SIGN_SUPPORTS:
        for signs in itertools.product((-1, 1), repeat=4):
            if signs[local_position] != required_sign:
                continue
            row = np.zeros(DIMENSION, dtype=np.int64)
            for idx, sign in zip(support, signs):
                row[idx] = sign
            rows.append(row)
    core = np.asarray(rows, dtype=np.int64)
    assert core.shape == (496, DIMENSION)
    return core


def decode_growth_scaffold() -> np.ndarray:
    rows = []
    for item in json.loads(GROWTH_SCAFFOLD_JSON):
        row = np.zeros(DIMENSION, dtype=np.float64)
        if item[0] == 'a':
            row[int(item[1])] = 2 if item[2] == '+' else -2
        else:
            support = [int(x) for x in item[1]]
            signs = [1 if ch == '+' else -1 for ch in item[2]]
            for idx, sign in zip(support, signs):
                row[idx] = sign
        rows.append(row)
    scaffold = np.asarray(rows, dtype=np.float64)
    assert scaffold.shape == (104, DIMENSION)
    return scaffold


def pair_metrics(core_unit: np.ndarray, remainder_unit: np.ndarray) -> dict[str, object]:
    core = normalize_rows(core_unit)
    rem = normalize_rows(remainder_unit)
    cr = core @ rem.T
    rr_i, rr_j = np.triu_indices(rem.shape[0], 1)
    rr = np.sum(rem[rr_i] * rem[rr_j], axis=1)
    values = np.concatenate([cr.reshape(-1), rr])
    return {
        'N': int(core.shape[0] + rem.shape[0]),
        'core_count': int(core.shape[0]),
        'remainder_count': int(rem.shape[0]),
        'max_inner_product': float(np.max(values)),
        'max_excess': float(np.max(values) - THRESHOLD),
        'violating_pair_count': int(np.count_nonzero(values > THRESHOLD)),
        'cr_max_inner_product': float(np.max(cr)),
        'cr_max_excess': float(np.max(cr) - THRESHOLD),
        'cr_violating_pair_count': int(np.count_nonzero(cr > THRESHOLD)),
        'rr_max_inner_product': float(np.max(rr)),
        'rr_max_excess': float(np.max(rr) - THRESHOLD),
        'rr_violating_pair_count': int(np.count_nonzero(rr > THRESHOLD)),
    }


In [2]:
#@title Remainder optimization
@dataclass
class Phase:
    name: str
    target: float
    steps: int
    lr: float
    beta: float
    temp: float
    hinge_weight: float
    top_weight: float
    max_weight: float
    score_weight: float
    eval_every: int


class FixedCoreOptimizer:
    def __init__(self, core_unit: np.ndarray, initial_remainder: np.ndarray):
        from jax import config as jax_config
        jax_config.update('jax_enable_x64', True)
        import jax
        import jax.numpy as jnp

        self.jax = jax
        self.jnp = jnp
        self.core = jnp.asarray(normalize_rows(core_unit), dtype=jnp.float64)
        self.r = jnp.asarray(normalize_rows(initial_remainder), dtype=jnp.float64)
        rr_i, rr_j = np.triu_indices(initial_remainder.shape[0], 1)
        self.rr_i = jnp.asarray(rr_i, dtype=jnp.int32)
        self.rr_j = jnp.asarray(rr_j, dtype=jnp.int32)
        self.top_k = int(min(4096, core_unit.shape[0] * initial_remainder.shape[0] + len(rr_i)))
        self.m = jnp.zeros_like(self.r)
        self.v = jnp.zeros_like(self.r)
        self.t = 0
        self._step = self._build_step()

    def current(self) -> np.ndarray:
        return np.asarray(self.r, dtype=np.float64)

    def reset_adam(self) -> None:
        self.m = self.jnp.zeros_like(self.r)
        self.v = self.jnp.zeros_like(self.r)
        self.t = 0

    def _build_step(self):
        jax = self.jax
        jnp = self.jnp
        core = self.core
        rr_i = self.rr_i
        rr_j = self.rr_j
        top_k = self.top_k

        @jax.jit
        def step_fn(r, m, v, t, params):
            target, lr, beta, temp, hinge_weight, top_weight, max_weight, score_weight = params

            def loss_fn(raw):
                x = raw / jnp.linalg.norm(raw, axis=1, keepdims=True)
                cr = (core @ x.T).reshape(-1)
                rr = jnp.sum(x[rr_i] * x[rr_j], axis=1)
                dots = jnp.concatenate([cr, rr])
                excess = dots - target
                soft = jax.nn.softplus(beta * excess) / beta
                hinge = jnp.mean(soft**2)
                top_values = jax.lax.top_k(excess, top_k)[0]
                top_soft = jax.nn.softplus(beta * top_values) / beta
                top_lp = jnp.mean(top_soft**8) ** (1.0 / 8.0)
                centered = jnp.max(top_values)
                smooth_max = centered + temp * jnp.log(jnp.sum(jnp.exp((top_values - centered) / temp)))
                max_pos = jax.nn.softplus(beta * smooth_max) / beta
                clipped = jnp.clip(dots, -1.0, 1.0)
                distances = jnp.sqrt(jnp.maximum(1.0e-12, 8.0 * (1.0 - clipped)))
                score = jnp.sum(jnp.maximum(0.0, 2.0 - distances))
                return hinge_weight * hinge + top_weight * top_lp + max_weight * max_pos + score_weight * score

            value, grad = jax.value_and_grad(loss_fn)(r)
            unit = r / jnp.linalg.norm(r, axis=1, keepdims=True)
            tangent = grad - jnp.sum(grad * unit, axis=1, keepdims=True) * unit
            tangent_norm = jnp.linalg.norm(tangent, axis=1, keepdims=True)
            tangent = tangent / jnp.maximum(1.0, tangent_norm / 10.0)
            beta1 = 0.9
            beta2 = 0.999
            t_float = t.astype(jnp.float64)
            m2 = beta1 * m + (1.0 - beta1) * tangent
            v2 = beta2 * v + (1.0 - beta2) * tangent * tangent
            m_hat = m2 / (1.0 - beta1**t_float)
            v_hat = v2 / (1.0 - beta2**t_float)
            step = lr * m_hat / (jnp.sqrt(v_hat) + 1.0e-8)
            step_norm = jnp.linalg.norm(step, axis=1, keepdims=True)
            step = step / jnp.maximum(1.0, step_norm / 0.08)
            r2 = r - step
            r2 = r2 / jnp.linalg.norm(r2, axis=1, keepdims=True)
            return r2, m2, v2, value

        return step_fn

    def run_phase(self, phase: Phase, core_np: np.ndarray) -> tuple[np.ndarray, dict[str, object]]:
        params = self.jnp.asarray([
            phase.target, phase.lr, phase.beta, phase.temp,
            phase.hinge_weight, phase.top_weight, phase.max_weight, phase.score_weight,
        ], dtype=self.jnp.float64)
        best = self.current()
        best_metrics = pair_metrics(core_np, best)
        for step_idx in range(1, phase.steps + 1):
            self.t += 1
            self.r, self.m, self.v, _ = self._step(
                self.r, self.m, self.v, self.jnp.asarray(self.t, dtype=self.jnp.int32), params
            )
            if step_idx % phase.eval_every == 0 or step_idx == phase.steps:
                current = self.current()
                metrics = pair_metrics(core_np, current)
                if (metrics['max_excess'], metrics['violating_pair_count']) < (
                    best_metrics['max_excess'], best_metrics['violating_pair_count']
                ):
                    best = current.copy()
                    best_metrics = metrics
        self.r = self.jnp.asarray(best, dtype=self.jnp.float64)
        return best, best_metrics


def relax_remainder(core_unit: np.ndarray, scaffold: np.ndarray, seed: int = 0) -> tuple[np.ndarray, dict[str, object]]:
    rng = np.random.default_rng(seed)
    initial = normalize_rows(scaffold + 0.01 * rng.normal(size=scaffold.shape))
    phases = [
        Phase('coarse', 0.5, 1200, 9.0e-4, 80.0, 0.004, 95.0, 24.0, 18.0, 0.50, 200),
        Phase('focus', 0.5, 1600, 5.8e-4, 140.0, 0.0015, 150.0, 52.0, 36.0, 0.90, 200),
        Phase('slack1', 0.499995, 1200, 3.5e-4, 190.0, 0.00085, 195.0, 72.0, 52.0, 1.08, 200),
        Phase('slack2', 0.49999, 1000, 2.4e-4, 230.0, 0.00052, 225.0, 90.0, 64.0, 1.18, 200),
        Phase('final', 0.5, 1000, 1.5e-4, 250.0, 0.00042, 200.0, 105.0, 76.0, 1.28, 200),
    ]
    opt = FixedCoreOptimizer(core_unit, initial)
    best = initial
    best_metrics = pair_metrics(core_unit, best)
    for phase_index, phase in enumerate(phases):
        current, metrics = opt.run_phase(phase, core_unit)
        if (metrics['max_excess'], metrics['violating_pair_count']) < (
            best_metrics['max_excess'], best_metrics['violating_pair_count']
        ):
            best = current.copy()
            best_metrics = metrics
        if phase_index in (1, 3):
            opt.reset_adam()
    return normalize_rows(best), best_metrics


In [3]:
#@title Active-boundary repair and exact verification
def tangent_bases(vectors: np.ndarray) -> np.ndarray:
    bases = []
    eye = np.eye(DIMENSION, dtype=np.float64)
    for unit in normalize_rows(vectors):
        cols = []
        for e in eye:
            v = e - float(e @ unit) * unit
            for b in cols:
                v = v - float(v @ b) * b
            n = float(np.linalg.norm(v))
            if n > 1.0e-10:
                cols.append(v / n)
            if len(cols) == DIMENSION - 1:
                break
        if len(cols) != DIMENSION - 1:
            raise RuntimeError('failed to build tangent basis')
        bases.append(np.stack(cols, axis=1))
    return np.stack(bases, axis=0)


def chebyshev_repair(core_unit: np.ndarray, remainder_unit: np.ndarray) -> tuple[np.ndarray, dict[str, object]]:
    from scipy.optimize import linprog

    best = normalize_rows(remainder_unit)
    best_metrics = pair_metrics(core_unit, best)
    rem_n = best.shape[0]
    thresholds = [0.49998, 0.49995, 0.49990, 0.49985, 0.49980]
    step_bounds = [5.0e-6, 1.0e-5, 2.0e-5, 5.0e-5, 1.0e-4, 2.0e-4]

    for threshold in thresholds:
        for step_bound in step_bounds:
            current = normalize_rows(best)
            basis = tangent_bases(current)
            cr = core_unit @ current.T
            rr_i, rr_j = np.triu_indices(rem_n, 1)
            rr = np.sum(current[rr_i] * current[rr_j], axis=1)
            active_cr = list(zip(*np.where(cr >= threshold)))
            active_rr = [int(idx) for idx in np.where(rr >= threshold)[0]]
            if not active_cr and not active_rr:
                continue
            var_count = rem_n * (DIMENSION - 1)
            rows = []
            rhs = []
            for i, j in active_cr:
                row = np.zeros(var_count + 1, dtype=np.float64)
                jj = int(j)
                row[jj * 10:jj * 10 + 10] = core_unit[int(i)] @ basis[jj]
                row[-1] = -1.0
                rows.append(row)
                rhs.append(float(THRESHOLD - cr[int(i), jj]))
            for idx in active_rr:
                a = int(rr_i[idx])
                b = int(rr_j[idx])
                row = np.zeros(var_count + 1, dtype=np.float64)
                row[a * 10:a * 10 + 10] = current[b] @ basis[a]
                row[b * 10:b * 10 + 10] = current[a] @ basis[b]
                row[-1] = -1.0
                rows.append(row)
                rhs.append(float(THRESHOLD - rr[idx]))

            objective = np.zeros(var_count + 1, dtype=np.float64)
            objective[-1] = 1.0
            result = linprog(
                objective,
                A_ub=np.vstack(rows),
                b_ub=np.asarray(rhs, dtype=np.float64),
                bounds=[(-step_bound, step_bound)] * var_count + [(-2.0e-3, 2.0e-3)],
                method='highs',
                options={'time_limit': 10.0},
            )
            if not result.success:
                continue
            coords = np.asarray(result.x[:-1], dtype=np.float64).reshape(rem_n, DIMENSION - 1)
            delta = np.einsum('rjk,rk->rj', basis, coords)
            for scale in [1.5, 1.25, 1.0, 0.75, 0.5, 0.25, 0.125]:
                candidate = normalize_rows(current + scale * delta)
                metrics = pair_metrics(core_unit, candidate)
                if (metrics['max_excess'], metrics['violating_pair_count']) < (
                    best_metrics['max_excess'], best_metrics['violating_pair_count']
                ):
                    best = candidate
                    best_metrics = metrics
    return best, best_metrics


def dot_int(u: Iterable[int], v: Iterable[int]) -> int:
    return sum(int(a) * int(b) for a, b in zip(u, v))


def squared_norm_int(u: Iterable[int]) -> int:
    return sum(int(a) * int(a) for a in u)


def classify_rows(vectors: list[list[int]]) -> dict[str, int]:
    counts = Counter()
    for row in vectors:
        nonzero = [int(x) for x in row if int(x) != 0]
        if len(nonzero) == 1 and abs(nonzero[0]) == 2:
            counts['axis_core'] += 1
        elif len(nonzero) == 4 and all(abs(x) == 1 for x in nonzero):
            counts['weight4_core'] += 1
        else:
            counts['rounded_remainder'] += 1
    return dict(counts)


def verify_kissing_configuration(vectors: list[list[int]], dimension: int = DIMENSION) -> dict[str, object]:
    rows = [[int(x) for x in row] for row in vectors]
    if any(len(row) != dimension for row in rows):
        raise AssertionError('wrong dimension')
    if any(all(x == 0 for x in row) for row in rows):
        raise AssertionError('zero row')

    norms = [squared_norm_int(row) for row in rows]
    conflict_count = 0
    contact_count = 0
    positive_dot_pairs = 0
    min_margin = None
    min_positive_margin = None

    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            d = dot_int(rows[i], rows[j])
            if d <= 0:
                continue
            positive_dot_pairs += 1
            margin = norms[i] * norms[j] - 4 * d * d
            if min_margin is None or margin < min_margin:
                min_margin = margin
            if margin == 0:
                contact_count += 1
            elif margin > 0 and (min_positive_margin is None or margin < min_positive_margin):
                min_positive_margin = margin
            elif margin < 0:
                conflict_count += 1

    unit = np.asarray(rows, dtype=np.float64)
    unit = unit / np.linalg.norm(unit, axis=1, keepdims=True)
    gram = unit @ unit.T
    np.fill_diagonal(gram, -np.inf)
    max_cosine = float(np.max(gram))

    return {
        'num_spheres': len(rows),
        'dimension': dimension,
        'row_classes': classify_rows(rows),
        'conflict_count': int(conflict_count),
        'contact_count': int(contact_count),
        'positive_dot_pairs': int(positive_dot_pairs),
        'min_margin': min_margin,
        'min_positive_margin': min_positive_margin,
        'max_float_cosine': max_cosine,
        'min_center_distance': math.sqrt(max(0.0, 8.0 * (1.0 - max_cosine))),
        'valid': conflict_count == 0,
    }


def round_remainder_and_verify(core_int: np.ndarray, remainder_unit: np.ndarray, q: int = ROUNDING_DENOMINATOR) -> tuple[list[list[int]], dict[str, object]]:
    remainder_int = np.rint(normalize_rows(remainder_unit) * float(q)).astype(np.int64)
    full = np.vstack([core_int.astype(np.int64), remainder_int])
    vectors = [[int(x) for x in row] for row in full.tolist()]
    metrics = verify_kissing_configuration(vectors)
    return vectors, metrics


In [4]:
#@title Independently generate and verify a 600-point configuration
core_int = construct_core()
core_unit = normalize_rows(core_int.astype(np.float64))
scaffold = decode_growth_scaffold()

scaffold_metrics = pair_metrics(core_unit, scaffold)
relaxed_remainder, relaxed_metrics = relax_remainder(core_unit, scaffold, seed=0)
repaired_remainder, repaired_metrics = chebyshev_repair(core_unit, relaxed_remainder)
independent_vectors, independent_metrics = round_remainder_and_verify(core_int, repaired_remainder)

assert scaffold_metrics['violating_pair_count'] > 0
assert relaxed_metrics['violating_pair_count'] > 0
assert repaired_metrics['violating_pair_count'] == 0
assert independent_metrics['num_spheres'] == 600
assert independent_metrics['dimension'] == 11
assert independent_metrics['conflict_count'] == 0
assert independent_metrics['valid'] is True

print(json.dumps({
    'scaffold': scaffold_metrics,
    'after_relaxation': relaxed_metrics,
    'after_active_repair': repaired_metrics,
    'exact_integer_certificate': independent_metrics,
}, indent=2, sort_keys=True))


{
  "after_active_repair": {
    "N": 600,
    "core_count": 496,
    "cr_max_excess": -1.6705377836878643e-05,
    "cr_max_inner_product": 0.4999832946221631,
    "cr_violating_pair_count": 0,
    "max_excess": -1.664805248924317e-05,
    "max_inner_product": 0.49998335194751076,
    "remainder_count": 104,
    "rr_max_excess": -1.664805248924317e-05,
    "rr_max_inner_product": 0.49998335194751076,
    "rr_violating_pair_count": 0,
    "violating_pair_count": 0
  },
  "after_relaxation": {
    "N": 600,
    "core_count": 496,
    "cr_max_excess": 4.6884832253568653e-05,
    "cr_max_inner_product": 0.5000468848322536,
    "cr_violating_pair_count": 13,
    "max_excess": 4.7918521601686415e-05,
    "max_inner_product": 0.5000479185216017,
    "remainder_count": 104,
    "rr_max_excess": 4.7918521601686415e-05,
    "rr_max_inner_product": 0.5000479185216017,
    "rr_violating_pair_count": 3,
    "violating_pair_count": 16
  },
  "exact_integer_certificate": {
    "conflict_count": 0,
  

## Exact Station Certificate

The following data is the final verified $600$-point construction found by Station. The next cell verifies this certificate with the same exact integer verifier used above.

In [5]:
#@title Exact 600-point config from Station
STATION_EXACT_CONFIG_SHA256 = 'ef855571cb46ac940b7252f8e1b58874faebedda4e291882ee446abb7da50daa'
STATION_EXACT_CONFIG_JSON = r'''[[-2,0,0,0,0,0,0,0,0,0,0],[-1,-1,-1,0,0,0,0,0,0,0,-1],[-1,-1,-1,0,0,0,0,0,0,0,1],[-1,-1,0,0,-1,0,0,0,0,-1,0],[-1,-1,0,0,-1,0,0,0,0,1,0],[-1,-1,0,0,0,-1,0,-1,0,0,0],[-1,-1,0,0,0,-1,0,1,0,0,0],[-1,-1,0,0,0,1,0,-1,0,0,0],[-1,-1,0,0,0,1,0,1,0,0,0],[-1,-1,0,0,1,0,0,0,0,-1,0],[-1,-1,0,0,1,0,0,0,0,1,0],[-1,-1,1,0,0,0,0,0,0,0,-1],[-1,-1,1,0,0,0,0,0,0,0,1],[-1,0,-1,-1,0,0,0,-1,0,0,0],[-1,0,-1,-1,0,0,0,1,0,0,0],[-1,0,-1,0,-1,0,-1,0,0,0,0],[-1,0,-1,0,-1,0,1,0,0,0,0],[-1,0,-1,0,0,-1,0,0,-1,0,0],[-1,0,-1,0,0,0,0,0,1,-1,0],[-1,0,-1,0,0,0,0,0,1,1,0],[-1,0,-1,0,0,1,0,0,-1,0,0],[-1,0,-1,0,1,0,-1,0,0,0,0],[-1,0,-1,0,1,0,1,0,0,0,0],[-1,0,-1,1,0,0,0,-1,0,0,0],[-1,0,-1,1,0,0,0,1,0,0,0],[-1,0,0,-1,-1,0,0,0,0,0,-1],[-1,0,0,-1,-1,0,0,0,0,0,1],[-1,0,0,-1,0,-1,0,0,0,-1,0],[-1,0,0,-1,0,-1,0,0,0,1,0],[-1,0,0,-1,0,1,0,0,0,-1,0],[-1,0,0,-1,0,1,0,0,0,1,0],[-1,0,0,-1,1,0,0,0,0,0,-1],[-1,0,0,-1,1,0,0,0,0,0,1],[-1,0,0,0,-1,-1,0,0,1,0,0],[-1,0,0,0,-1,0,0,-1,-1,0,0],[-1,0,0,0,-1,0,0,1,-1,0,0],[-1,0,0,0,-1,1,0,0,1,0,0],[-1,0,0,0,0,-1,-1,0,0,0,-1],[-1,0,0,0,0,-1,-1,0,0,0,1],[-1,0,0,0,0,-1,1,0,0,0,-1],[-1,0,0,0,0,-1,1,0,0,0,1],[-1,0,0,0,0,0,-1,-1,0,-1,0],[-1,0,0,0,0,0,-1,-1,0,1,0],[-1,0,0,0,0,0,-1,1,0,-1,0],[-1,0,0,0,0,0,-1,1,0,1,0],[-1,0,0,0,0,0,0,-1,1,0,-1],[-1,0,0,0,0,0,0,-1,1,0,1],[-1,0,0,0,0,0,0,0,-1,-1,-1],[-1,0,0,0,0,0,0,0,-1,-1,1],[-1,0,0,0,0,0,0,0,-1,1,-1],[-1,0,0,0,0,0,0,0,-1,1,1],[-1,0,0,0,0,0,0,1,1,0,-1],[-1,0,0,0,0,0,0,1,1,0,1],[-1,0,0,0,0,0,1,-1,0,-1,0],[-1,0,0,0,0,0,1,-1,0,1,0],[-1,0,0,0,0,0,1,1,0,-1,0],[-1,0,0,0,0,0,1,1,0,1,0],[-1,0,0,0,0,1,-1,0,0,0,-1],[-1,0,0,0,0,1,-1,0,0,0,1],[-1,0,0,0,0,1,1,0,0,0,-1],[-1,0,0,0,0,1,1,0,0,0,1],[-1,0,0,0,1,-1,0,0,1,0,0],[-1,0,0,0,1,0,0,-1,-1,0,0],[-1,0,0,0,1,0,0,1,-1,0,0],[-1,0,0,0,1,1,0,0,1,0,0],[-1,0,0,1,-1,0,0,0,0,0,-1],[-1,0,0,1,-1,0,0,0,0,0,1],[-1,0,0,1,0,-1,0,0,0,-1,0],[-1,0,0,1,0,-1,0,0,0,1,0],[-1,0,0,1,0,1,0,0,0,-1,0],[-1,0,0,1,0,1,0,0,0,1,0],[-1,0,0,1,1,0,0,0,0,0,-1],[-1,0,0,1,1,0,0,0,0,0,1],[-1,0,1,-1,0,0,0,-1,0,0,0],[-1,0,1,-1,0,0,0,1,0,0,0],[-1,0,1,0,-1,0,-1,0,0,0,0],[-1,0,1,0,-1,0,1,0,0,0,0],[-1,0,1,0,0,-1,0,0,-1,0,0],[-1,0,1,0,0,0,0,0,1,-1,0],[-1,0,1,0,0,0,0,0,1,1,0],[-1,0,1,0,0,1,0,0,-1,0,0],[-1,0,1,0,1,0,-1,0,0,0,0],[-1,0,1,0,1,0,1,0,0,0,0],[-1,0,1,1,0,0,0,-1,0,0,0],[-1,0,1,1,0,0,0,1,0,0,0],[-1,1,-1,0,0,0,0,0,0,0,-1],[-1,1,-1,0,0,0,0,0,0,0,1],[-1,1,0,0,-1,0,0,0,0,-1,0],[-1,1,0,0,-1,0,0,0,0,1,0],[-1,1,0,0,0,-1,0,-1,0,0,0],[-1,1,0,0,0,-1,0,1,0,0,0],[-1,1,0,0,0,1,0,-1,0,0,0],[-1,1,0,0,0,1,0,1,0,0,0],[-1,1,0,0,1,0,0,0,0,-1,0],[-1,1,0,0,1,0,0,0,0,1,0],[-1,1,1,0,0,0,0,0,0,0,-1],[-1,1,1,0,0,0,0,0,0,0,1],[0,-1,-1,-1,0,0,0,0,0,-1,0],[0,-1,-1,-1,0,0,0,0,0,1,0],[0,-1,-1,0,-1,-1,0,0,0,0,0],[0,-1,-1,0,-1,1,0,0,0,0,0],[0,-1,-1,0,1,-1,0,0,0,0,0],[0,-1,-1,0,1,1,0,0,0,0,0],[0,-1,-1,1,0,0,0,0,0,-1,0],[0,-1,-1,1,0,0,0,0,0,1,0],[0,-1,0,-1,-1,0,0,-1,0,0,0],[0,-1,0,-1,-1,0,0,1,0,0,0],[0,-1,0,-1,0,-1,0,0,0,0,-1],[0,-1,0,-1,0,-1,0,0,0,0,1],[0,-1,0,-1,0,1,0,0,0,0,-1],[0,-1,0,-1,0,1,0,0,0,0,1],[0,-1,0,-1,1,0,0,-1,0,0,0],[0,-1,0,-1,1,0,0,1,0,0,0],[0,-1,0,0,0,0,0,-1,0,-1,-1],[0,-1,0,0,0,0,0,-1,0,-1,1],[0,-1,0,0,0,0,0,-1,0,1,-1],[0,-1,0,0,0,0,0,-1,0,1,1],[0,-1,0,0,0,0,0,1,0,-1,-1],[0,-1,0,0,0,0,0,1,0,-1,1],[0,-1,0,0,0,0,0,1,0,1,-1],[0,-1,0,0,0,0,0,1,0,1,1],[0,-1,0,1,-1,0,0,-1,0,0,0],[0,-1,0,1,-1,0,0,1,0,0,0],[0,-1,0,1,0,-1,0,0,0,0,-1],[0,-1,0,1,0,-1,0,0,0,0,1],[0,-1,0,1,0,1,0,0,0,0,-1],[0,-1,0,1,0,1,0,0,0,0,1],[0,-1,0,1,1,0,0,-1,0,0,0],[0,-1,0,1,1,0,0,1,0,0,0],[0,-1,1,-1,0,0,0,0,0,-1,0],[0,-1,1,-1,0,0,0,0,0,1,0],[0,-1,1,0,-1,-1,0,0,0,0,0],[0,-1,1,0,-1,1,0,0,0,0,0],[0,-1,1,0,1,-1,0,0,0,0,0],[0,-1,1,0,1,1,0,0,0,0,0],[0,-1,1,1,0,0,0,0,0,-1,0],[0,-1,1,1,0,0,0,0,0,1,0],[0,0,-2,0,0,0,0,0,0,0,0],[0,0,-1,-1,-1,0,0,0,1,0,0],[0,0,-1,-1,0,-1,-1,0,0,0,0],[0,0,-1,-1,0,-1,1,0,0,0,0],[0,0,-1,-1,0,0,0,0,-1,0,-1],[0,0,-1,-1,0,0,0,0,-1,0,1],[0,0,-1,-1,0,1,-1,0,0,0,0],[0,0,-1,-1,0,1,1,0,0,0,0],[0,0,-1,-1,1,0,0,0,1,0,0],[0,0,-1,0,-1,0,0,-1,0,0,-1],[0,0,-1,0,-1,0,0,-1,0,0,1],[0,0,-1,0,-1,0,0,0,-1,-1,0],[0,0,-1,0,-1,0,0,0,-1,1,0],[0,0,-1,0,-1,0,0,1,0,0,-1],[0,0,-1,0,-1,0,0,1,0,0,1],[0,0,-1,0,0,-1,0,-1,0,-1,0],[0,0,-1,0,0,-1,0,-1,0,1,0],[0,0,-1,0,0,-1,0,0,1,0,-1],[0,0,-1,0,0,-1,0,0,1,0,1],[0,0,-1,0,0,-1,0,1,0,-1,0],[0,0,-1,0,0,-1,0,1,0,1,0],[0,0,-1,0,0,0,-1,0,0,-1,-1],[0,0,-1,0,0,0,-1,0,0,-1,1],[0,0,-1,0,0,0,-1,0,0,1,-1],[0,0,-1,0,0,0,-1,0,0,1,1],[0,0,-1,0,0,0,1,0,0,-1,-1],[0,0,-1,0,0,0,1,0,0,-1,1],[0,0,-1,0,0,0,1,0,0,1,-1],[0,0,-1,0,0,0,1,0,0,1,1],[0,0,-1,0,0,1,0,-1,0,-1,0],[0,0,-1,0,0,1,0,-1,0,1,0],[0,0,-1,0,0,1,0,0,1,0,-1],[0,0,-1,0,0,1,0,0,1,0,1],[0,0,-1,0,0,1,0,1,0,-1,0],[0,0,-1,0,0,1,0,1,0,1,0],[0,0,-1,0,1,0,0,-1,0,0,-1],[0,0,-1,0,1,0,0,-1,0,0,1],[0,0,-1,0,1,0,0,0,-1,-1,0],[0,0,-1,0,1,0,0,0,-1,1,0],[0,0,-1,0,1,0,0,1,0,0,-1],[0,0,-1,0,1,0,0,1,0,0,1],[0,0,-1,1,-1,0,0,0,1,0,0],[0,0,-1,1,0,-1,-1,0,0,0,0],[0,0,-1,1,0,-1,1,0,0,0,0],[0,0,-1,1,0,0,0,0,-1,0,-1],[0,0,-1,1,0,0,0,0,-1,0,1],[0,0,-1,1,0,1,-1,0,0,0,0],[0,0,-1,1,0,1,1,0,0,0,0],[0,0,-1,1,1,0,0,0,1,0,0],[0,0,0,-2,0,0,0,0,0,0,0],[0,0,0,-1,-1,-1,0,0,-1,0,0],[0,0,0,-1,-1,0,-1,0,0,-1,0],[0,0,0,-1,-1,0,-1,0,0,1,0],[0,0,0,-1,-1,0,1,0,0,-1,0],[0,0,0,-1,-1,0,1,0,0,1,0],[0,0,0,-1,-1,1,0,0,-1,0,0],[0,0,0,-1,0,-1,0,-1,1,0,0],[0,0,0,-1,0,-1,0,1,1,0,0],[0,0,0,-1,0,0,-1,-1,0,0,-1],[0,0,0,-1,0,0,-1,-1,0,0,1],[0,0,0,-1,0,0,-1,1,0,0,-1],[0,0,0,-1,0,0,-1,1,0,0,1],[0,0,0,-1,0,0,0,-1,-1,-1,0],[0,0,0,-1,0,0,0,-1,-1,1,0],[0,0,0,-1,0,0,0,0,1,-1,-1],[0,0,0,-1,0,0,0,0,1,-1,1],[0,0,0,-1,0,0,0,0,1,1,-1],[0,0,0,-1,0,0,0,0,1,1,1],[0,0,0,-1,0,0,0,1,-1,-1,0],[0,0,0,-1,0,0,0,1,-1,1,0],[0,0,0,-1,0,0,1,-1,0,0,-1],[0,0,0,-1,0,0,1,-1,0,0,1],[0,0,0,-1,0,0,1,1,0,0,-1],[0,0,0,-1,0,0,1,1,0,0,1],[0,0,0,-1,0,1,0,-1,1,0,0],[0,0,0,-1,0,1,0,1,1,0,0],[0,0,0,-1,1,-1,0,0,-1,0,0],[0,0,0,-1,1,0,-1,0,0,-1,0],[0,0,0,-1,1,0,-1,0,0,1,0],[0,0,0,-1,1,0,1,0,0,-1,0],[0,0,0,-1,1,0,1,0,0,1,0],[0,0,0,-1,1,1,0,0,-1,0,0],[0,0,0,0,-2,0,0,0,0,0,0],[0,0,0,0,-1,-1,-1,-1,0,0,0],[0,0,0,0,-1,-1,-1,1,0,0,0],[0,0,0,0,-1,-1,0,0,0,-1,-1],[0,0,0,0,-1,-1,0,0,0,-1,1],[0,0,0,0,-1,-1,0,0,0,1,-1],[0,0,0,0,-1,-1,0,0,0,1,1],[0,0,0,0,-1,-1,1,-1,0,0,0],[0,0,0,0,-1,-1,1,1,0,0,0],[0,0,0,0,-1,0,0,-1,1,-1,0],[0,0,0,0,-1,0,0,-1,1,1,0],[0,0,0,0,-1,0,0,1,1,-1,0],[0,0,0,0,-1,0,0,1,1,1,0],[0,0,0,0,-1,1,-1,-1,0,0,0],[0,0,0,0,-1,1,-1,1,0,0,0],[0,0,0,0,-1,1,0,0,0,-1,-1],[0,0,0,0,-1,1,0,0,0,-1,1],[0,0,0,0,-1,1,0,0,0,1,-1],[0,0,0,0,-1,1,0,0,0,1,1],[0,0,0,0,-1,1,1,-1,0,0,0],[0,0,0,0,-1,1,1,1,0,0,0],[0,0,0,0,0,-2,0,0,0,0,0],[0,0,0,0,0,-1,0,-1,-1,0,-1],[0,0,0,0,0,-1,0,-1,-1,0,1],[0,0,0,0,0,-1,0,1,-1,0,-1],[0,0,0,0,0,-1,0,1,-1,0,1],[0,0,0,0,0,0,0,-2,0,0,0],[0,0,0,0,0,0,0,0,0,-2,0],[0,0,0,0,0,0,0,0,0,0,-2],[0,0,0,0,0,0,0,0,0,0,2],[0,0,0,0,0,0,0,0,0,2,0],[0,0,0,0,0,0,0,2,0,0,0],[0,0,0,0,0,1,0,-1,-1,0,-1],[0,0,0,0,0,1,0,-1,-1,0,1],[0,0,0,0,0,1,0,1,-1,0,-1],[0,0,0,0,0,1,0,1,-1,0,1],[0,0,0,0,0,2,0,0,0,0,0],[0,0,0,0,1,-1,-1,-1,0,0,0],[0,0,0,0,1,-1,-1,1,0,0,0],[0,0,0,0,1,-1,0,0,0,-1,-1],[0,0,0,0,1,-1,0,0,0,-1,1],[0,0,0,0,1,-1,0,0,0,1,-1],[0,0,0,0,1,-1,0,0,0,1,1],[0,0,0,0,1,-1,1,-1,0,0,0],[0,0,0,0,1,-1,1,1,0,0,0],[0,0,0,0,1,0,0,-1,1,-1,0],[0,0,0,0,1,0,0,-1,1,1,0],[0,0,0,0,1,0,0,1,1,-1,0],[0,0,0,0,1,0,0,1,1,1,0],[0,0,0,0,1,1,-1,-1,0,0,0],[0,0,0,0,1,1,-1,1,0,0,0],[0,0,0,0,1,1,0,0,0,-1,-1],[0,0,0,0,1,1,0,0,0,-1,1],[0,0,0,0,1,1,0,0,0,1,-1],[0,0,0,0,1,1,0,0,0,1,1],[0,0,0,0,1,1,1,-1,0,0,0],[0,0,0,0,1,1,1,1,0,0,0],[0,0,0,0,2,0,0,0,0,0,0],[0,0,0,1,-1,-1,0,0,-1,0,0],[0,0,0,1,-1,0,-1,0,0,-1,0],[0,0,0,1,-1,0,-1,0,0,1,0],[0,0,0,1,-1,0,1,0,0,-1,0],[0,0,0,1,-1,0,1,0,0,1,0],[0,0,0,1,-1,1,0,0,-1,0,0],[0,0,0,1,0,-1,0,-1,1,0,0],[0,0,0,1,0,-1,0,1,1,0,0],[0,0,0,1,0,0,-1,-1,0,0,-1],[0,0,0,1,0,0,-1,-1,0,0,1],[0,0,0,1,0,0,-1,1,0,0,-1],[0,0,0,1,0,0,-1,1,0,0,1],[0,0,0,1,0,0,0,-1,-1,-1,0],[0,0,0,1,0,0,0,-1,-1,1,0],[0,0,0,1,0,0,0,0,1,-1,-1],[0,0,0,1,0,0,0,0,1,-1,1],[0,0,0,1,0,0,0,0,1,1,-1],[0,0,0,1,0,0,0,0,1,1,1],[0,0,0,1,0,0,0,1,-1,-1,0],[0,0,0,1,0,0,0,1,-1,1,0],[0,0,0,1,0,0,1,-1,0,0,-1],[0,0,0,1,0,0,1,-1,0,0,1],[0,0,0,1,0,0,1,1,0,0,-1],[0,0,0,1,0,0,1,1,0,0,1],[0,0,0,1,0,1,0,-1,1,0,0],[0,0,0,1,0,1,0,1,1,0,0],[0,0,0,1,1,-1,0,0,-1,0,0],[0,0,0,1,1,0,-1,0,0,-1,0],[0,0,0,1,1,0,-1,0,0,1,0],[0,0,0,1,1,0,1,0,0,-1,0],[0,0,0,1,1,0,1,0,0,1,0],[0,0,0,1,1,1,0,0,-1,0,0],[0,0,0,2,0,0,0,0,0,0,0],[0,0,1,-1,-1,0,0,0,1,0,0],[0,0,1,-1,0,-1,-1,0,0,0,0],[0,0,1,-1,0,-1,1,0,0,0,0],[0,0,1,-1,0,0,0,0,-1,0,-1],[0,0,1,-1,0,0,0,0,-1,0,1],[0,0,1,-1,0,1,-1,0,0,0,0],[0,0,1,-1,0,1,1,0,0,0,0],[0,0,1,-1,1,0,0,0,1,0,0],[0,0,1,0,-1,0,0,-1,0,0,-1],[0,0,1,0,-1,0,0,-1,0,0,1],[0,0,1,0,-1,0,0,0,-1,-1,0],[0,0,1,0,-1,0,0,0,-1,1,0],[0,0,1,0,-1,0,0,1,0,0,-1],[0,0,1,0,-1,0,0,1,0,0,1],[0,0,1,0,0,-1,0,-1,0,-1,0],[0,0,1,0,0,-1,0,-1,0,1,0],[0,0,1,0,0,-1,0,0,1,0,-1],[0,0,1,0,0,-1,0,0,1,0,1],[0,0,1,0,0,-1,0,1,0,-1,0],[0,0,1,0,0,-1,0,1,0,1,0],[0,0,1,0,0,0,-1,0,0,-1,-1],[0,0,1,0,0,0,-1,0,0,-1,1],[0,0,1,0,0,0,-1,0,0,1,-1],[0,0,1,0,0,0,-1,0,0,1,1],[0,0,1,0,0,0,1,0,0,-1,-1],[0,0,1,0,0,0,1,0,0,-1,1],[0,0,1,0,0,0,1,0,0,1,-1],[0,0,1,0,0,0,1,0,0,1,1],[0,0,1,0,0,1,0,-1,0,-1,0],[0,0,1,0,0,1,0,-1,0,1,0],[0,0,1,0,0,1,0,0,1,0,-1],[0,0,1,0,0,1,0,0,1,0,1],[0,0,1,0,0,1,0,1,0,-1,0],[0,0,1,0,0,1,0,1,0,1,0],[0,0,1,0,1,0,0,-1,0,0,-1],[0,0,1,0,1,0,0,-1,0,0,1],[0,0,1,0,1,0,0,0,-1,-1,0],[0,0,1,0,1,0,0,0,-1,1,0],[0,0,1,0,1,0,0,1,0,0,-1],[0,0,1,0,1,0,0,1,0,0,1],[0,0,1,1,-1,0,0,0,1,0,0],[0,0,1,1,0,-1,-1,0,0,0,0],[0,0,1,1,0,-1,1,0,0,0,0],[0,0,1,1,0,0,0,0,-1,0,-1],[0,0,1,1,0,0,0,0,-1,0,1],[0,0,1,1,0,1,-1,0,0,0,0],[0,0,1,1,0,1,1,0,0,0,0],[0,0,1,1,1,0,0,0,1,0,0],[0,0,2,0,0,0,0,0,0,0,0],[0,1,-1,-1,0,0,0,0,0,-1,0],[0,1,-1,-1,0,0,0,0,0,1,0],[0,1,-1,0,-1,-1,0,0,0,0,0],[0,1,-1,0,-1,1,0,0,0,0,0],[0,1,-1,0,1,-1,0,0,0,0,0],[0,1,-1,0,1,1,0,0,0,0,0],[0,1,-1,1,0,0,0,0,0,-1,0],[0,1,-1,1,0,0,0,0,0,1,0],[0,1,0,-1,-1,0,0,-1,0,0,0],[0,1,0,-1,-1,0,0,1,0,0,0],[0,1,0,-1,0,-1,0,0,0,0,-1],[0,1,0,-1,0,-1,0,0,0,0,1],[0,1,0,-1,0,1,0,0,0,0,-1],[0,1,0,-1,0,1,0,0,0,0,1],[0,1,0,-1,1,0,0,-1,0,0,0],[0,1,0,-1,1,0,0,1,0,0,0],[0,1,0,0,0,0,0,-1,0,-1,-1],[0,1,0,0,0,0,0,-1,0,-1,1],[0,1,0,0,0,0,0,-1,0,1,-1],[0,1,0,0,0,0,0,-1,0,1,1],[0,1,0,0,0,0,0,1,0,-1,-1],[0,1,0,0,0,0,0,1,0,-1,1],[0,1,0,0,0,0,0,1,0,1,-1],[0,1,0,0,0,0,0,1,0,1,1],[0,1,0,1,-1,0,0,-1,0,0,0],[0,1,0,1,-1,0,0,1,0,0,0],[0,1,0,1,0,-1,0,0,0,0,-1],[0,1,0,1,0,-1,0,0,0,0,1],[0,1,0,1,0,1,0,0,0,0,-1],[0,1,0,1,0,1,0,0,0,0,1],[0,1,0,1,1,0,0,-1,0,0,0],[0,1,0,1,1,0,0,1,0,0,0],[0,1,1,-1,0,0,0,0,0,-1,0],[0,1,1,-1,0,0,0,0,0,1,0],[0,1,1,0,-1,-1,0,0,0,0,0],[0,1,1,0,-1,1,0,0,0,0,0],[0,1,1,0,1,-1,0,0,0,0,0],[0,1,1,0,1,1,0,0,0,0,0],[0,1,1,1,0,0,0,0,0,-1,0],[0,1,1,1,0,0,0,0,0,1,0],[1,-1,-1,0,0,0,0,0,0,0,-1],[1,-1,-1,0,0,0,0,0,0,0,1],[1,-1,0,0,-1,0,0,0,0,-1,0],[1,-1,0,0,-1,0,0,0,0,1,0],[1,-1,0,0,0,-1,0,-1,0,0,0],[1,-1,0,0,0,-1,0,1,0,0,0],[1,-1,0,0,0,1,0,-1,0,0,0],[1,-1,0,0,0,1,0,1,0,0,0],[1,-1,0,0,1,0,0,0,0,-1,0],[1,-1,0,0,1,0,0,0,0,1,0],[1,-1,1,0,0,0,0,0,0,0,-1],[1,-1,1,0,0,0,0,0,0,0,1],[1,0,-1,-1,0,0,0,-1,0,0,0],[1,0,-1,-1,0,0,0,1,0,0,0],[1,0,-1,0,-1,0,-1,0,0,0,0],[1,0,-1,0,-1,0,1,0,0,0,0],[1,0,-1,0,0,-1,0,0,-1,0,0],[1,0,-1,0,0,0,0,0,1,-1,0],[1,0,-1,0,0,0,0,0,1,1,0],[1,0,-1,0,0,1,0,0,-1,0,0],[1,0,-1,0,1,0,-1,0,0,0,0],[1,0,-1,0,1,0,1,0,0,0,0],[1,0,-1,1,0,0,0,-1,0,0,0],[1,0,-1,1,0,0,0,1,0,0,0],[1,0,0,-1,-1,0,0,0,0,0,-1],[1,0,0,-1,-1,0,0,0,0,0,1],[1,0,0,-1,0,-1,0,0,0,-1,0],[1,0,0,-1,0,-1,0,0,0,1,0],[1,0,0,-1,0,1,0,0,0,-1,0],[1,0,0,-1,0,1,0,0,0,1,0],[1,0,0,-1,1,0,0,0,0,0,-1],[1,0,0,-1,1,0,0,0,0,0,1],[1,0,0,0,-1,-1,0,0,1,0,0],[1,0,0,0,-1,0,0,-1,-1,0,0],[1,0,0,0,-1,0,0,1,-1,0,0],[1,0,0,0,-1,1,0,0,1,0,0],[1,0,0,0,0,-1,-1,0,0,0,-1],[1,0,0,0,0,-1,-1,0,0,0,1],[1,0,0,0,0,-1,1,0,0,0,-1],[1,0,0,0,0,-1,1,0,0,0,1],[1,0,0,0,0,0,-1,-1,0,-1,0],[1,0,0,0,0,0,-1,-1,0,1,0],[1,0,0,0,0,0,-1,1,0,-1,0],[1,0,0,0,0,0,-1,1,0,1,0],[1,0,0,0,0,0,0,-1,1,0,-1],[1,0,0,0,0,0,0,-1,1,0,1],[1,0,0,0,0,0,0,0,-1,-1,-1],[1,0,0,0,0,0,0,0,-1,-1,1],[1,0,0,0,0,0,0,0,-1,1,-1],[1,0,0,0,0,0,0,0,-1,1,1],[1,0,0,0,0,0,0,1,1,0,-1],[1,0,0,0,0,0,0,1,1,0,1],[1,0,0,0,0,0,1,-1,0,-1,0],[1,0,0,0,0,0,1,-1,0,1,0],[1,0,0,0,0,0,1,1,0,-1,0],[1,0,0,0,0,0,1,1,0,1,0],[1,0,0,0,0,1,-1,0,0,0,-1],[1,0,0,0,0,1,-1,0,0,0,1],[1,0,0,0,0,1,1,0,0,0,-1],[1,0,0,0,0,1,1,0,0,0,1],[1,0,0,0,1,-1,0,0,1,0,0],[1,0,0,0,1,0,0,-1,-1,0,0],[1,0,0,0,1,0,0,1,-1,0,0],[1,0,0,0,1,1,0,0,1,0,0],[1,0,0,1,-1,0,0,0,0,0,-1],[1,0,0,1,-1,0,0,0,0,0,1],[1,0,0,1,0,-1,0,0,0,-1,0],[1,0,0,1,0,-1,0,0,0,1,0],[1,0,0,1,0,1,0,0,0,-1,0],[1,0,0,1,0,1,0,0,0,1,0],[1,0,0,1,1,0,0,0,0,0,-1],[1,0,0,1,1,0,0,0,0,0,1],[1,0,1,-1,0,0,0,-1,0,0,0],[1,0,1,-1,0,0,0,1,0,0,0],[1,0,1,0,-1,0,-1,0,0,0,0],[1,0,1,0,-1,0,1,0,0,0,0],[1,0,1,0,0,-1,0,0,-1,0,0],[1,0,1,0,0,0,0,0,1,-1,0],[1,0,1,0,0,0,0,0,1,1,0],[1,0,1,0,0,1,0,0,-1,0,0],[1,0,1,0,1,0,-1,0,0,0,0],[1,0,1,0,1,0,1,0,0,0,0],[1,0,1,1,0,0,0,-1,0,0,0],[1,0,1,1,0,0,0,1,0,0,0],[1,1,-1,0,0,0,0,0,0,0,-1],[1,1,-1,0,0,0,0,0,0,0,1],[1,1,0,0,-1,0,0,0,0,-1,0],[1,1,0,0,-1,0,0,0,0,1,0],[1,1,0,0,0,-1,0,-1,0,0,0],[1,1,0,0,0,-1,0,1,0,0,0],[1,1,0,0,0,1,0,-1,0,0,0],[1,1,0,0,0,1,0,1,0,0,0],[1,1,0,0,1,0,0,0,0,-1,0],[1,1,0,0,1,0,0,0,0,1,0],[1,1,1,0,0,0,0,0,0,0,-1],[1,1,1,0,0,0,0,0,0,0,1],[2,0,0,0,0,0,0,0,0,0,0],[-49680,-19303,7,-49913,-68,303,-49329,-47,-47270,-70,299],[-49814,-49426,49,-49940,6,101,-23890,-79,44842,-33,208],[-49706,-49401,-95,49978,-205,98,41778,-89,-29079,-132,-80],[-49815,46996,-67,-49974,-87,23,-45107,-97,27887,-137,77],[-49652,19569,1,-49916,-94,313,49326,-62,47189,-104,319],[-49651,-13001,-31,49903,-293,68,-49355,-30,-49391,-68,-120],[-49723,49395,-101,49960,-211,86,-41956,-96,28833,-141,-52],[-49,-98775,-325,-50,297,-83,-11051,252,-11004,7,-300],[-36,-49883,-49760,-36,-36,17,49805,-49751,-8938,7,-17],[23,-46590,-49890,-2,-10,222,49440,49781,-20438,5,260],[56,-45593,-49901,55,-6,153,-22359,49797,49511,-89,196],[65,-44431,-4,58,-49965,-37,49560,-178,-24367,-116,-49794],[-22,-48973,-363,-22,-49655,-24,27882,303,-43549,-24,49617],[182,-21722,-42,102,-8,-49752,-49058,-159,-46381,49949,237],[-36,-49596,195,-35,-88,-49755,47622,-99,-17528,-49895,120],[271,-45551,297,81,-58,-49603,-25585,-81,48082,49979,311],[127,-49611,12,12,-85,-49830,44590,-33,-23951,49938,128],[-164,-41470,-36,91,-37,49523,-49601,-36,-30254,-49518,-2],[-24,-49785,-32,-32,-40,49658,10409,-46,49738,-49732,2],[-210,-49490,-193,171,61,49771,-11985,-210,49497,49799,80],[-818,-30994,-36,781,-39,48442,49186,-36,-43056,-49173,616],[270,-21500,58,96,49986,365,-46893,-161,-48753,-2,-49630],[270,-49689,-2,-56,49800,55,39647,31,-31539,-23,49842],[48537,-31730,1323,48531,-40,739,-43043,602,49256,-25,781],[-49634,13382,-31,49889,-325,91,49281,-50,49393,-126,-94],[-33,-9197,49777,-35,55,-34,49748,-49860,49767,-45,-117],[10,-43367,49912,195,46,-80,48474,49830,-28200,-140,189],[-70,-49048,49895,211,169,-135,-15415,49602,49056,-333,201],[-23,-31478,-49675,-23,52,40,-39808,-49908,-49660,34,66],[-38,39792,-49781,-39,-60,218,31445,-49815,-49680,44,33],[266,-49272,41,82,-49945,147,-21020,-281,46549,-103,-49688],[-30,-48977,-381,-29,-49634,-29,-43562,315,27910,-32,49607],[72,25252,76,65,-49956,195,45994,-139,47677,46,-49781],[66,-15278,-258,-11,-49716,36,49109,152,49012,-105,49809],[-39,-8462,-36,-42,-6,-243,-99256,-37,8749,233,-75],[-257,-8483,-37,-257,-3,-41,8835,-39,-99246,-4,-3],[-230,8258,-38,-230,-3,-41,-8762,-38,99272,-40,-59],[-39,8273,-36,-42,-6,-221,99275,-37,-8720,142,-58],[189,-47393,93,22,49950,203,-25991,-52,45841,-9,-49819],[221,-49700,-32,-101,49769,54,-31538,33,39628,-34,49879],[296,21513,38,82,49980,388,46734,-212,48937,-9,-49598],[60,8397,-36,-125,49794,-37,49801,-36,49822,-36,49876],[-34,9192,49792,-35,80,-33,-49791,-49835,-49735,-22,-75],[-24,-49692,49587,-24,284,-24,31448,-49651,-40217,-23,-344],[-42,31424,-49706,-42,20,54,39755,-49944,49669,85,125],[66,45431,-49901,61,-20,166,22622,49807,-49532,-67,161],[20,46519,-49884,8,-13,202,-49553,49779,20346,25,263],[238,49338,22,74,-49945,123,20662,-271,-46618,-108,-49708],[-38,49041,-383,-37,-49640,-34,43625,313,-27667,-39,49618],[52,-24478,94,57,16,-49798,-44367,-81,-49569,-49954,215],[294,45706,311,80,-67,-49585,25656,-82,-47909,49984,319],[-33,49563,180,-34,-93,-49759,-47577,-94,17706,-49904,136],[125,49642,17,21,-99,-49837,-44683,-49,23741,49917,132],[-27,49750,-29,-31,-40,49703,-10012,-43,-49830,-49711,2],[-206,49500,-187,170,41,49782,11683,-191,-49553,49794,73],[-154,41119,-36,80,-38,49552,49594,-36,30512,-49631,-3],[-80,41341,-130,262,88,49829,-49432,-207,29721,49809,172],[205,47597,96,17,49933,203,26000,-55,-45635,-13,-49825],[248,49686,-30,-77,49782,26,-39642,-7,31585,-43,49838],[-37,49643,49611,-36,243,-36,-31567,-49666,40136,-38,-379],[-68,49029,49879,188,171,-142,15302,49627,-49102,-304,171],[35,43335,49903,171,51,-91,-48453,49845,28274,-130,163],[-52,98676,-289,-52,465,-83,11429,217,11479,13,-541],[49764,-32354,-126,-49798,-36,-220,-49808,-133,-38936,-171,3],[49827,-49924,-31,-49870,-36,-92,-7151,-31,49866,-93,-2],[49354,-41285,166,49357,-38,-62,-30910,183,-49680,-3,-4],[49838,49857,-38,-49911,-37,-107,7154,-38,-49881,-45,3],[49785,32286,-143,-49784,4,-207,49799,-121,38996,-192,6],[49727,49830,-31,49755,-39,-25,-49909,-42,-8820,-5,-2],[49383,40952,174,49408,-38,-54,31095,164,49760,-49,-23],[-151,30028,-39,218,61,49870,49421,-310,41180,49728,153],[61,-25418,72,61,-49961,178,-45967,-114,-47551,8,-49837],[67,44262,-17,54,-49960,-47,-49618,-168,24562,-119,-49796],[212,-48004,48,64,49973,223,46418,-186,-23880,-47,-49733],[-29,-49695,49578,-29,284,-29,-40272,-49619,31438,-30,-407],[186,44106,270,-47,-156,-49627,28922,38,-47509,-49980,156],[-36,49827,-49771,-36,-36,14,-49886,-49712,8951,75,-44],[-37,49644,49607,-36,249,-37,40143,-49689,-31528,-34,-310],[-12054,57108,-8821,-6959,4244,-17522,-52170,8559,-54757,4684,-18081],[-49763,-49220,3,49938,-234,123,-29286,-37,41826,15,-36],[-79,27240,49961,88,63,-55,49445,49807,42852,-150,88],[48572,31409,480,48589,-39,812,43103,1296,-49318,-37,818],[-45,-27300,49981,102,59,-38,-49445,49809,-42789,-154,96],[-77,-41180,-122,261,70,49833,49451,-201,-29900,49813,174],[-39,28241,-49915,7,63,67,45386,49886,46502,28,97],[194,49699,-45,-119,49753,51,31583,26,-39607,-37,49884],[-49757,49217,-8,49929,-252,141,29103,-39,-41976,7,-14],[-128,-29790,-18,183,86,49907,-49394,-284,-41294,49766,147],[205,48132,63,59,49949,241,-46438,-159,23582,-31,-49756],[-49797,49415,71,-49932,5,135,23712,-91,-44976,-39,240],[-752,30650,-36,778,-40,48559,-49244,-35,43119,-49162,607],[-42,-28239,-49933,10,60,60,-45460,49881,-46417,-8,100],[108,-8495,3,-131,49831,-37,-49805,3,-49808,3,49831],[49712,-49866,-33,49709,-39,-29,49844,-44,9313,-53,2],[174,-44201,249,-41,-152,-49639,-28890,46,47425,-49983,165],[173,21536,-74,121,9,-49780,49170,-191,46361,49910,237],[-22,-39847,-49758,-20,-68,237,-31499,-49770,49669,96,39],[-14747,-57243,-5434,-4914,3737,-10731,52274,11705,54408,11754,-18843],[49770,38945,-150,-49807,-14,-189,-49817,-144,32305,-188,-4],[56,24725,108,42,43,-49806,44251,-78,49588,-49908,224],[49803,-39019,-112,-49785,16,-190,49777,-106,-32261,-182,10],[-35,49043,-340,-34,-49668,-36,-27573,301,43640,-36,49628],[-49824,-47013,-72,-49974,-91,44,44985,-86,-28041,-93,54],[51,14813,-252,15,-49739,81,-49182,196,-49120,-46,49748]]'''

station_vectors = json.loads(STATION_EXACT_CONFIG_JSON)
assert __import__('hashlib').sha256(STATION_EXACT_CONFIG_JSON.encode()).hexdigest() == STATION_EXACT_CONFIG_SHA256
station_metrics = verify_kissing_configuration(station_vectors)

assert station_metrics['num_spheres'] == 600
assert station_metrics['dimension'] == 11
assert station_metrics['conflict_count'] == 0
assert station_metrics['valid'] is True
assert 'independent_vectors' in globals()
assert independent_vectors != station_vectors

print(json.dumps(station_metrics, indent=2, sort_keys=True))


{
  "conflict_count": 0,
  "contact_count": 17024,
  "dimension": 11,
  "max_float_cosine": 0.5,
  "min_center_distance": 2.0,
  "min_margin": 0,
  "min_positive_margin": 12,
  "num_spheres": 600,
  "positive_dot_pairs": 65737,
  "row_classes": {
    "axis_core": 16,
    "rounded_remainder": 104,
    "weight4_core": 480
  },
  "valid": true
}
